### If you get an error here, it means you need to run the prepare_data.ipynb file first

In [1]:
import json
import numpy as np
import pandas as pd

from rnanorm import TMM
from pathlib import Path


description_path = Path('description.json')
if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    data = json.load(f)

## Preprocessing steps

1. **Load data** — Read the dataset and split columns into clinical variables and raw gene-level counts.

2. **Robust-expression filter (defined on raw counts)** — Keep genes with the 75th percentile of raw counts across samples \(\ge 100\) (i.e., expressed at \(\ge 100\) counts in at least 25% of samples).

3. **TMM normalization** — Apply TMM to the full gene count matrix to compute normalization factors correcting for library size and RNA composition; obtain normalized counts.

4. **Apply gene filter and log-transform** — Subset to the robustly expressed genes and apply \(\log(x+1)\).

5. <strike>**Z-score standardization** — For each gene, center and scale across samples to mean 0 and standard deviation 1.</strike>

In [2]:
data_file_name = data['data_file_names'][0]

num_clinical_features = data['data_summary']['clinical_features']

df = pd.read_csv(data_file_name)


clinical_cols = df.columns[:num_clinical_features]
gene_cols = df.columns[num_clinical_features:]

raw_counts = df[gene_cols]
print('Before preprocessing:\n - n_genes: %d' % (df[gene_cols].shape[1]))

# create mask of robustly expressed genes
robus_genes_mask = np.percentile(df[gene_cols], 75, axis=0) >= 100

# normalize counts
df[gene_cols] = TMM().set_output(transform="pandas").fit_transform(df[gene_cols])

# apply robust gene mask
gene_cols = df[gene_cols].columns[robus_genes_mask].tolist()
df = df[list(clinical_cols) + gene_cols]

# # log transform and standardize (genes only)
df[gene_cols] = np.log(df[gene_cols] + 1)
# df[gene_cols] = (df[gene_cols] - df[gene_cols].mean(axis=0)) / df[gene_cols].std(axis=0)

print("\nAfter preprocessing:\n - n_genes: %d" % (df[gene_cols].shape[1]))

transformed_data_file_name = data_file_name.replace("_raw_counts_NCBI", "_preprocessed")


Before preprocessing:
 - n_genes: 39376



After preprocessing:
 - n_genes: 14203


In [3]:
url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
df_gene_annotations = df_gene_annotations[df_gene_annotations["Status"] == "active"]
df_gene_annotations = df_gene_annotations[df_gene_annotations["GeneType"] == "protein-coding"]
df_gene_annotations = df_gene_annotations.reset_index(drop=True)

protein_coding_genes = set(df_gene_annotations["Symbol"].dropna())
gene_cols = [gene for gene in gene_cols if gene in protein_coding_genes]
df = df[list(clinical_cols) + gene_cols]

print("\nAfter protein-coding filter:\n - n_genes: %d" % len(gene_cols))

df.to_csv(transformed_data_file_name, index=False)

if transformed_data_file_name not in data['data_file_names']:
    data['data_file_names'].append(transformed_data_file_name)

with open(description_path, 'w') as f:
    json.dump(data, f, indent=4)

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_21310/4029965179.py:2: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(



After protein-coding filter:
 - n_genes: 13078


## Survival variables encodings

Authors encode survival variables in various ways, here we have just a check if all the survival variables are already numeric. If they are not, we convert them by hand.


In [4]:
df_ = df.copy()

endpoints = []
for endpoint in data["survival-endpoints"]:
    endpoint_time = endpoint["time_var"]["var_name"]
    endpoint_event = endpoint["event_var"]["var_name"]

    if endpoint_time == "unknown" or endpoint_event == "unknown":
        continue

    # Replace "Not available" and similar missing-value strings with NaN
    for col in [endpoint_time, endpoint_event]:
        df_[col] = df_[col].replace(["Not available", "NA", "N/A", "not available", "-"], pd.NA)


    # make sure time and event are numeric
    df_[endpoint_time] = pd.to_numeric(df_[endpoint_time])
    df_[endpoint_event] = pd.to_numeric(df_[endpoint_event])
    
    endpoints.append((endpoint_time, endpoint_event))

# Flatten endpoints and use it to order survival columns
surv_cols = [col for tup in endpoints for col in tup]
other_cols = [col for col in df_.columns if col not in surv_cols]
ordered_cols = surv_cols + other_cols

df_[ordered_cols].to_csv(transformed_data_file_name, index=False)

In [5]:
df[ordered_cols]

,survival time,survival,sample_title,tissue,gender,age,prognostic subtype,SAMD11,NOC2L,HES4,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,35.8,1.0,2004-950RNAseq,Brain Metastatic Tissue,Male,67.0,Poor,2.254931,4.540211,0.293703,...,6.281492,2.964554,4.949571,7.379952,4.789904,3.788943,4.996821,4.581002,2.818548,4.917973
1,1.0,0.0,2006-262-RNAseq,Brain Metastatic Tissue,Male,55.0,Good,2.600297,3.406464,1.033125,...,7.439998,4.657450,6.926805,8.227490,6.421354,5.732441,7.375770,6.918782,4.549217,6.839389
2,1.1,1.0,2006-76RNASeq,Brain Metastatic Tissue,Male,61.0,Good,2.121720,3.574719,2.551187,...,6.074428,2.635651,4.962224,7.092147,4.835275,4.285369,5.572828,5.513111,3.375943,5.612652
3,3.2,1.0,2007-109RNASeq,Brain Metastatic Tissue,Female,61.0,Good,1.446262,3.401710,2.985803,...,6.902861,3.201464,5.671691,8.078788,5.544196,4.731804,6.312369,5.672232,3.493351,6.291637
4,2.6,1.0,2007-263-RNAseq,Brain Metastatic Tissue,Female,61.0,Good,3.515690,3.857159,1.242937,...,8.280964,5.136008,7.612852,8.958045,6.251670,6.483668,8.213153,7.970868,5.070564,7.805208
5,8.6,1.0,2007-45RNAseq,Brain Metastatic Tissue,Female,45.0,Cluster 1,1.956078,3.812501,2.892563,...,6.610917,2.924615,5.095985,7.393207,5.011815,3.997627,5.281810,5.017451,2.938537,5.237363
6,58.0,0.0,2007-550RNAseq,Brain Metastatic Tissue,Female,81.0,Poor,1.952617,4.124452,1.834337,...,6.844973,3.746257,5.778427,7.782623,5.808101,4.529742,5.959016,5.666299,3.587226,6.005397
7,113.2,0.0,2007-563RNAseq,Brain Metastatic Tissue,Female,47.0,Poor,2.120248,4.185286,3.228017,...,6.259064,2.791758,5.223452,7.208025,4.574452,3.796107,5.310168,4.918311,2.786485,5.009356
8,10.0,1.0,2007-601RNAseq,Brain Metastatic Tissue,Female,47.0,Others,2.589278,3.695381,2.656078,...,6.255755,2.380955,4.926914,6.984726,5.337730,3.683887,5.057682,4.748923,2.941459,5.174180
9,29.1,1.0,2008-1840RNAseq,Brain Metastatic Tissue,Male,55.0,Others,1.826524,3.850437,2.825001,...,6.593334,3.878646,6.223154,8.258997,5.554796,4.745881,6.089461,5.744612,3.536522,6.277023
